# 01. OpenAI API 기초
**SK하이닉스 Autonomous R&D — Day 3** 

> ChatGPT 웹과 **같은 모델**을 코드에서 호출.

---
## 0. 라이브러리 설치

아래 셀을 **최초 1회** 실행.

In [ ]:
!pip install openai==1.58.1

#노트북에 커널을 설정한 상태로 !뒤에 명령어를 넣으면(terminal 명령어) 해당 환경에서 terminal로 그 명령어를 실행한 것과 같은 결과를 얻는다.

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----------------------------------- ---- 1.8/2.1 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 9.0 MB/s  0:00:00

   -- -------------------------------------  1/15 [tqdm]
   -------- -------------------------------  3/15 [pydantic-core]
   ------------- --------------------------  5/15 [idna]
   ------------------ ---------------------  7/15 [distro]
   -------------------------- ------------- 10/15 [pydantic]
   -------------------------- ------------- 10/15 [pydantic]
   -------------------------- ------------- 10/15 [pydantic]
   -------------------------- ------------- 10/15 [pydantic]
   -------------------------- ------------- 10/15 [pydantic]
   ----------------------------- ---------- 11/15 [httpcore]
   ----------------------------- ---------- 11/15 [httpcore]
   -------------------------------- ------- 12/15 [anyio]
   ---------------------------------- ----- 13/15 [httpx

In [2]:
!pip install openai python-dotenv

---
## 1. LLM API란?

강의자료 요약:

| 구분 | ChatGPT 웹 | API |
|------|------------|-----|
| 사용자 | 사람이 직접 입력 | **프로그램**이 요청 |
| 결과 | 화면에 표시 | **response 객체**로 반환 |
| 본질 | 대화 | **Prompt → Text** |

API 요청의 핵심 3요소:
1. **`model`** — 어떤 LLM을 쓸지
2. **`messages`** — system / user (대화 내용)
3. **`temperature`** — 답변의 무작위성 (0=일관, 1=창의)

---
## 2. API 키 연결


처음에는 **키를 변수에 넣어** 바로 연결해 봅니다.

> OpenAI Platform → [API keys](https://platform.openai.com/api-keys) 에서 발급

In [ ]:
from openai import OpenAI

# 아래에 본인 키를 붙여넣고 실행
api_key = ''

client = OpenAI(api_key=api_key)

In [5]:
# 연결 테스트
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Hi, reply with one word: OK'}],
    max_tokens=10,
)
print('연결 성공:', test.choices[0].message.content)

연결 성공: OK


1. **`.env` 파일**에 키 저장 (코드와 분리) (OPENAI_API_KEY=sk...)
2. **`.gitignore`**에 `.env` 등록 → Git에 **절대 올리지 않음**

In [6]:
from pathlib import Path

root = Path('..')  # ttest/
env_path = root / '.env'
gitignore_path = root / '.gitignore'

print('.env 존재:', env_path.exists(), '→', env_path.resolve())
print('.gitignore 존재:', gitignore_path.exists())

if gitignore_path.exists():
    content = gitignore_path.read_text(encoding='utf-8')
    print('.env in .gitignore:', '.env' in content)

.env 존재: True → C:\Users\Admin\Desktop\실습\.env
.gitignore 존재: True
.env in .gitignore: True


In [7]:
from dotenv import load_dotenv
import os

load_dotenv(root / '.env')
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    raise ValueError('ttest/.env에 OPENAI_API_KEY=sk-... 를 설정하세요.')

client = OpenAI(api_key=api_key)
print('✅ .env에서 API 키 로드 완료 (코드에 키 없음)')

✅ .env에서 API 키 로드 완료 (코드에 키 없음)


---
## 3. 첫 API 호출 

`chat.completions.create()` — **messages** 리스트를 보내면 AI가 답변합니다.

| role | 역할 |
|------|------|
| `system` | AI 역할·규칙 (선택) |
| `user` | 사용자 질문 |

In [8]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user',   'content': '2002년 월드컵 우승 팀이 어디야?'},
    ],
)

In [21]:
response.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))

In [22]:
response.choices[0].message

ChatCompletionMessage(content='2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[])

In [23]:
response.choices[0].message.content

'2002년 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.'

In [ ]:
####### 2022년 월드컵 우승팀 물어보기
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.9,
    messages=[
        {'role' : 'system', 'content' : 'You are a helpful assistant.'},
        {'role' : 'user', 'content' : '2022년 월드컵 우승 팀이 어디야?'},
    ],
)


Ellipsis

In [26]:
response.choices[0].message.content

'2022년 월드컵 우승 팀은 아르헨티나입니다. 아르헨티나는 결승에서 프랑스를 상대로 승리하여 36년 만에 월드컵 트로피를 다시 차지했습니다. 결승 경기는 2022년 12월 18일에 카타르의 루사일 스타디움에서 열렸습니다.'

In [27]:
####### 2022년 월드컵 우승팀 물어보기
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role' : 'system', 'content' : 'You are a helpful assistant.'},
        {'role' : 'user', 'content' : '2022년 월드컵 우승 팀이 어디야?'},
    ],
)

In [28]:
response.choices[0].message.content

'2022년 FIFA 월드컵 우승 팀은 아르헨티나입니다. 아르헨티나는 결승에서 프랑스를 상대로 승리하여 36년 만에 월드컵 트로피를 차지했습니다.'

---
## 4. 응답(Response) 객체

API는 **텍스트만**이 아니라 **객체**를 돌려줍니다.

| 필드 | 의미 |
|------|------|
| `response.choices[0].message.content` | **실제 답변 텍스트** ← 가장 많이 사용 |
| `response.choices[0].finish_reason` | `stop`=정상 종료, `length`=토큰 초과 |
| `response.usage.total_tokens` | 이번 호출에 쓴 토큰 수 (비용 참고) |
| `response.id` | 요청 ID (로그·디버깅용) |

In [ ]:
print('ID:', response.id)
print('finish_reason:', response.choices[0].finish_reason)  #EOS token이 나오면 답변이 end된 것으로 판정한다.
print('답변:', response.choices[0].message.content)
print('토큰 사용:', response.usage.total_tokens, '(prompt:', response.usage.prompt_tokens,
      '/ completion:', response.usage.completion_tokens, ')')

ID: chatcmpl-DuBUr8f9JU4KmzM9z1Gd9NCo3EejW
finish_reason: stop
답변: 2022년 FIFA 월드컵 우승 팀은 아르헨티나입니다. 아르헨티나는 결승에서 프랑스를 상대로 승리하여 36년 만에 월드컵 트로피를 차지했습니다.
토큰 사용: 82 (prompt: 30 / completion: 52 )


---
## 5. System / User 프롬프트 

같은 `user` 질문이라도 **`system`에 역할·출력 규칙**을 주면 답변 **형식·깊이·관점**이 달라집니다.

아래는 **의도적으로 짧고 모호한 질문**을 사용합니다. system 유무에 따라 차이가 잘 보입니다.

| | system | user |
|---|--------|------|
| 역할 | 모델의 **역할·규칙·형식** | 이번 **질문·작업** |
| 비유 | 직무 기술서 | 오늘 업무 지시 |

In [30]:
# 질문을 짧고 모호하게 → system이 없으면 범용 답변, 있으면 역할·형식에 맞춘 답변
question = 'for문이 뭐야?'

r1 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[{'role': 'user', 'content': question}],
)

print('=== system 없음 (범용 설명, 길어질 수 있음) ===')
print(r1.choices[0].message.content)

=== system 없음 (범용 설명, 길어질 수 있음) ===
`for`문은 프로그래밍에서 반복적인 작업을 수행할 때 사용하는 제어 구조입니다. 특정 조건이 만족될 때까지 코드 블록을 반복 실행할 수 있게 해줍니다. 주로 리스트, 배열, 또는 범위와 같은 반복 가능한 객체를 순회할 때 사용됩니다.

예를 들어, Python에서 `for`문을 사용하는 기본적인 예시는 다음과 같습니다:

```python
# 0부터 4까지의 숫자를 출력
for i in range(5):
    print(i)
```

위의 코드에서 `range(5)`는 0부터 4까지의 숫자를 생성하고, `for`문은 이 숫자들을 하나씩 `i`에 할당하여 반복적으로 출력합니다.

다른 언어에서도 비슷한 구조를 가지고 있으며, 사용법은 약간씩 다를 수 있습니다. 예를 들어, JavaScript에서는 다음과 같이 사용할 수 있습니다:

```javascript
// 0부터 4까지의 숫자를 출력
for (let i = 0; i < 5; i++) {
    console.log(i);
}
```

이처럼 `for`문은 반복적인 작업을 간편하게 처리할 수 있도록 도와주는 유용한 도구입니다.


In [31]:
r2 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[
        {
            'role': 'system',
            'content': '''You are a Python tutor.
Rules:
- Answer in Korean only
- Use exactly 3 lines: [비유 1문장] / [코드 예시 max 3 lines] / [실무 한 줄]
- No long explanation, no bullet lists''',
        },
        {'role': 'user', 'content': question},
    ],
)

print('=== system 있음 (튜터 + 3줄 형식 강제) ===')
print(r2.choices[0].message.content)

=== system 있음 (튜터 + 3줄 형식 강제) ===
for문은 반복하는 기계처럼, 주어진 작업을 여러 번 수행하게 해줍니다.  
```python
for i in range(5):
    print(i)
```
반복적인 작업을 간편하게 처리할 수 있습니다.


---
## 6. temperature — **일반 예시**

같은 질문, 다른 `temperature` → 결과가 어떻게 달라지는지 확인합니다.

| temperature | 특징 | 용도 |
|-------------|------|------|
| 0 ~ 0.3 | 일관적, 사실 위주 | 분석, 판정, 코드 |
| 0.7 ~ 1.0 | 다양·창의적 | 브레인스토밍, 카피 |

In [32]:
question = '팀 워크숍 아이스브레이킹 아이디어 1가지만.'

#temp값은 일종의 무작위성이다. 다음에 올 확률이 가장 높은 단어를 택하는게 아니라 차선 이후를 선택한다. 높을수록 창의적인 답변을 얻는다.
for temp in [0.0, 0.7, 1.0]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temp,
        messages=[{'role': 'user', 'content': question}],
    )
    print(f'[temperature={temp}] {r.choices[0].message.content}')
    print()

[temperature=0.0] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 세 가지 사실을 말하는데, 그 중 두 가지는 진실이고 하나는 거짓입니다. 다른 팀원들은 어떤 것이 거짓인지 맞춰보는 방식입니다. 이 게임은 서로에 대한 이해를 높이고, 자연스럽게 대화를 유도하는 데 도움이 됩니다.

[temperature=0.7] 팀 워크숍에서 사용할 수 있는 아이스브레이킹 아이디어로 "두 진실과 한 거짓" 게임을 제안합니다.

### 방법:
1. 팀원 각자가 자신에 대한 두 가지 사실과 하나의 거짓을 준비합니다.
2. 차례로 각 팀원이 자신의 세 가지 정보(두 진실, 한 거짓)를 발표합니다.
3. 나머지 팀원들은 어떤 것이 거짓인지 맞추는 시간을 가집니다.
4. 정답을 맞춘 후, 각 팀원은 자신의 이야기를 조금 더 자세히 공유할 수 있습니다.

### 효과:
이 게임은 팀원들 간의 재미있는 대화를 유도하고, 서로에 대한 이해를 높이는 데 도움이 됩니다. 또한, 웃음을 통해 팀 분위기를 부드럽게 만들어 줍니다.

[temperature=1.0] "두 진실과 하나의 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 두 가지 진실과 하나의 거짓을 말합니다. 다른 팀원들은 어떤 것이 거짓인지 맞춰보는 게임입니다. 이 게임은 팀원 간의 친밀감을 높이고 서로에 대해 더 알 수 있는 기회를 제공합니다. 재미있고 간단하게 진행할 수 있어 아이스브레이킹에 적합합니다!



---
## 7. `max_tokens` — 출력 길이 제한

답변이 너무 길어지는 것을 막거나, 비용을 줄일 때 사용합니다.

In [33]:
for max_tok in [20, 100]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0.3,
        max_tokens=max_tok,
        messages=[{'role': 'user', 'content': '대한민국의 역사를 요약해줘.'}],
    )
    print(f'--- max_tokens={max_tok} (finish: {r.choices[0].finish_reason}) ---')
    print(r.choices[0].message.content)
    print()

--- max_tokens=20 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 포함

--- max_tokens=100 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 겪어왔습니다. 다음은 대한민국 역사의 주요 사건들을 요약한 것입니다.

1. **고대 및 삼국시대**: 
   - 한반도에는 기원전 2333년경 단군이 고조선을 세운 것으로 전해지며, 이후 고구려, 백제, 신라의 삼국이 형성되었습니다. 
   -



---
## 8. `ask()` 함수 — 재사용 패턴

같은 API 호출을 함수로 감싸기.

In [34]:
def ask(user_message, system_message='You are a helpful assistant.', temperature=0.2, max_tokens=None):
    """1턴 질의 — 답변 텍스트만 반환"""
    kwargs = dict(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_message},
            {'role': 'user',   'content': user_message},
        ],
    )
    if max_tokens is not None:
        kwargs['max_tokens'] = max_tokens
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


print(ask('내이름은 혜소니야 만나서 반가워', max_tokens=150))

안녕하세요, 혜소니님! 만나서 반가워요. 어떻게 도와드릴까요?


In [35]:
print(ask('내이름이 뭔지 알아?', max_tokens=150))

죄송하지만, 당신의 이름을 알 수 있는 방법이 없습니다. 당신의 이름을 알려주시면 그에 맞춰 대화할 수 있습니다!


---
## 9. 멀티턴 대화 

이전 대화를 `messages`에 **그대로 이어 붙이면** 후속 질문이 가능합니다.

```
system → user → assistant → user → ...
```

### 기존대화

In [36]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '2일차만 더 자세히. 맛집 2곳 포함.'},
]

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
제주도 2박 3일 여행 계획입니다.

**1일차:** 제주공항 도착 후 한라산 국립공원 탐방 및 성산일출봉에서 일몰 감상.

**2일차:** 우도 섬으로 가서 자전거 탐방 후, 협재 해수욕장에서 해수욕과 바비큐 즐기기.

**3일차:** 제주 민속촌 방문 후, 제주도 특산물 쇼핑 및 공항으로 귀환.

=== 2턴 (맥락 유지) ===
물론입니다! 2일차 일정을 자세히 설명해드릴게요.

### 2일차 일정

**아침:**
- **식사:** 지역 유명 카페에서 아침 식사. 추천 메뉴는 프렌치 토스트와 커피입니다. (예: "카페 블루" 또는 "브런치 하우스")

**오전:**
- **관광:** 시내 중심가에 위치한 유명한 박물관 방문. 예를 들어, "국립박물관"에서 지역의 역사와 문화를 배울 수 있습니다.

**점심:**
- **식사:** 현지 맛집에서 점심. 추천하는 곳은 "김밥천국"으로, 다양한 종류의 김밥과 떡볶이를 맛볼 수 있습니다.

**오후:**
- **관광:** 아름다운 공원이나 정원에서 산책. 예를 들어, "도심 속 공원"에서 자연을 느끼며 여유로운 시간을 보낼 수 있습니다.

**저녁:**
- **식사:** 저녁은 "전통 한식당"에서 한정식으로 즐기는 것을 추천합니다. (예: "한정식 집" 또는 "전통 한식당") 다양한 반찬과 함께 정갈한 한식을 경험할 수 있습니다.

**저녁 후:**
- **활동:** 지역의 야경을 즐길 수 있는 전망대나 강변 산책로에서 여유로운 시간을 보내세요.

이렇게 2일차 일정을 구성해보았습니다. 맛집 두 곳과 함께 다양한 활동을 즐길 수 있을 것입니다!


In [37]:
answer1

'제주도 2박 3일 여행 계획입니다.\n\n**1일차:** 제주공항 도착 후 한라산 국립공원 탐방 및 성산일출봉에서 일몰 감상.\n\n**2일차:** 우도 섬으로 가서 자전거 탐방 후, 협재 해수욕장에서 해수욕과 바비큐 즐기기.\n\n**3일차:** 제주 민속촌 방문 후, 제주도 특산물 쇼핑 및 공항으로 귀환.'

### 멀티턴

### Assistant 메시지는 앞서 있었던 prompt들(user/system)에 대한 모델의 응답을 구성하며, 종종 대화 상태를 유지하기 위해 히스토리의 일부로 저장됩니다.

#### 특징
모델이 생성한 응답을 포함한다.

이전 message의 context 처리를 위해 대화의 연속성 유지를 돕는다.

다음 API 호출에서는 assistant 메시지를 포함시켜야 문맥이 이어짐.

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages.append({'role': 'assistant', 'content': answer1})  #이전에 물어본 것에 대한 대답을 두번째 질문에 포함해야 맥락이 유지된다.
messages.append({'role': 'user', 'content': '2일차만 더 자세히. 맛집 2곳 포함.'})

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
**1일차:** 제주공항 도착 후, 한라산 국립공원에서 하이킹을 즐기고, 저녁에는 제주시의 동문시장에서 제주 전통 음식을 맛본다.

**2일차:** 성산일출봉에서 일출을 감상한 후, 우도 섬으로 배를 타고 가서 자전거를 타며 섬을 탐방하고, 저녁에는 해변 근처의 식당에서 신선한 해산물을 즐긴다.

**3일차:** 제주 민속촌에서 제주 전통 문화를 체험하고, 마지막으로 협재해수욕장에서 여유로운 시간을 보낸 후, 제주공항으로 돌아간다.

=== 2턴 (맥락 유지) ===
**2일차 상세 일정:**

- **아침:** 성산일출봉에서 일출 감상 후, 근처의 **'성산일출봉 해물뚝배기'**에서 신선한 해물뚝배기로 아침 식사를 즐깁니다.

- **오전:** 성산일출봉을 하산한 후, 우도 섬으로 가는 배를 탑니다. 우도에 도착하면 자전거를 대여하여 섬을 탐방합니다. 우도의 아름다운 해변과 경치를 즐기며 여유로운 시간을 보냅니다.

- **점심:** 우도에서 유명한 **'땅콩 아이스크림'**을 맛보며 간단한 간식을 즐기고, **'우도 뚱뚱이네'**에서 우도 땅콩을 사용한 땅콩국수로 점심을 해결합니다.

- **오후:** 우도의 주요 관광지를 둘러본 후, 배를 타고 제주 본섬으로 돌아옵니다. 이후 제주 서쪽의 협재해수욕장으로 이동하여 해변에서 수영이나 일광욕을 즐깁니다.

- **저녁:** 협재해수욕장 근처의 **'협재해물탕'**에서 신선한 해산물로 만든 해물탕을 맛보며 하루를 마무리합니다.

- **밤:** 저녁 식사 후, 제주 시내로 돌아가 여유롭게 산책하며 여행의 마지막 밤을 즐깁니다.


---
## ✏️ 실습 1

아래 주제로 **2턴** 대화를 만들어 보세요.

1턴: "Python for문 기본 문법 설명해줘"
2턴: "방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘"

In [39]:
!pip install streamlit

  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl.metadata (59 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.1 MB 932.9 kB/s eta 0:00:11
   -- ------------------------------------- 0.5/10.1 MB 932.9 kB/s eta 0:00:11
   -- ------------------------------------- 0.5/10.1 MB 932.9 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/10.1 MB 610.3 kB/s eta 0:00:16
   ---- ----------------------------------- 1.0/10.1 MB 699.0 kB/s eta 0:00:13
   ----- ---------------------------------- 1.3/10.1 MB 780.8 kB/s eta 0:00:12
   ----- ---------------------------------- 1.3/10.1 MB 780.8 kB/s eta 0:00:12
   ------- -------------------------------- 1.8/10.1 MB 883.1 kB/s eta 0:00:10
   -------- --------

In [ ]:
# ── 여기에 작성 ──
messages = [
    {'role': 'system', 'content': 'You are a Python tutor. Answer in Korean.'},
    # {'role': 'user', 'content': '...'},
]
# r1 = client.chat.completions.create(...)
# messages.append(...)
# r2 = client.chat.completions.create(...)

---
## ✏️ 실습 1

커서 프롬프트 : 3일차 폴더에 스트림릿과 실습코드의 openai api를 이용해서 간단한 챗봇을 만드는 python 코드 만들어줘

In [ ]:
!pip install streamlit